# Capítulo 5: Análise Geográfica de Clientes

Neste notebook, vamos analisar a distribuição geográfica dos clientes por **Cidade e Estado**, e sua relação com as personas de consumo identificadas no Capítulo 3.

## 1. Preparação dos Dados

Carregamento dos dados originais de clientes e dos dados de personas previamente clusterizados.

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df_clientes = pd.read_csv('../data/df_clientes.csv')
df_personas = pd.read_csv('../data/df_clustered_final.csv')

print("Dados de clientes:", df_clientes.shape)
print("Dados de personas:", df_personas.shape)

### 1.1 Enriquecimento de Dados (Simulação de Cidades)

Caso o dataset original não possua cidades, vamos simular uma distribuição para fins de análise.

In [ ]:
if 'cidade' not in df_clientes.columns:
    print("⚠️ Coluna 'cidade' não encontrada. Gerando dados sintéticos para demonstração...")
    mapeamento_cidades = {
        "SP": ["São Paulo", "Campinas", "Santos", "Ribeirão Preto"],
        "RJ": ["Rio de Janeiro", "Niterói", "Búzios", "Petrópolis"],
        "MG": ["Belo Horizonte", "Uberlândia", "Ouro Preto", "Tiradentes"],
        "PR": ["Curitiba", "Londrina", "Maringá", "Foz do Iguaçu"],
        "SC": ["Florianópolis", "Joinville", "Blumenau", "Balneário Camboriú"],
        "RS": ["Porto Alegre", "Gramado", "Caxias do Sul", "Canoas"],
        "BA": ["Salvador", "Porto Seguro", "Ilhéus", "Feira de Santana"],
        "GO": ["Goiânia", "Anápolis", "Pirenópolis", "Caldas Novas"]
    }
    df_clientes['cidade'] = df_clientes['estado'].apply(lambda x: np.random.choice(mapeamento_cidades.get(x, ["Outra"])))
    print("✅ Coluna 'cidade' adicionada!")

## 2. Merge & Limpeza

Unificar os dados de localização com os rótulos de persona, corrigindo divergências de tipos de dados nos IDs.

In [ ]:
# Padronizando o cliente_id para garantir que ambos sejam do mesmo tipo (String sem prefixo)
df_clientes['cliente_id_search'] = df_clientes['cliente_id'].astype(str).str.replace('CUST-', '')
df_personas['cliente_id_search'] = df_personas['cliente_id'].astype(str)

# Realizando o merge usando o ID normalizado
df_geo = pd.merge(df_clientes, df_personas, left_on='cliente_id_search', right_on='cliente_id_search', how='inner')

# Limpando colunas duplicadas ou auxiliares
df_geo = df_geo.drop(columns=['cliente_id_search', 'cliente_id_y'])
df_geo = df_geo.rename(columns={'cliente_id_x': 'cliente_id'})

print("Dados após o merge:", df_geo.shape)
df_geo.head()

## 3. Análise por Estado

Visualização da contagem geral de clientes por estado.

In [ ]:
contagem_estado = df_geo['estado'].value_counts().reset_index()
contagem_estado.columns = ['Estado', 'Total_Clientes']

fig = px.bar(contagem_estado, x='Estado', y='Total_Clientes', 
             title='Quantidade de Clientes por Estado',
             color='Total_Clientes', color_continuous_scale='Viridis')
fig.show()

## 4. Análise por Cidade

Explorando a distribuição municipal e hierárquica.

In [ ]:
# Top 10 Cidades com mais clientes
top_cidades = df_geo['cidade'].value_counts().head(10).reset_index()
top_cidades.columns = ['Cidade', 'Total_Clientes']

fig_city = px.bar(top_cidades, x='Total_Clientes', y='Cidade', 
                  orientation='h', title='Top 10 Cidades por Número de Clientes',
                  color='Total_Clientes', color_continuous_scale='Blues')
fig_city.show()

# Visualização Hierárquica (Treemap)
fig_tree = px.treemap(df_geo, path=['estado', 'cidade'], 
                      title='Distribuição Hierárquica: Estado -> Cidade')
fig_tree.show()

## 5. Cruzamento Persona vs Geografia

Identificar onde estão concentrados os clientes de cada Persona, especialmente os "Campeões".

In [ ]:
persona_geo = df_geo.groupby(['estado', 'Persona']).size().reset_index(name='Quantidade')

fig_persona = px.sunburst(persona_geo, path=['estado', 'Persona'], values='Quantidade',
                          title='Proporção de Personas por Estado (Visão Sunburst)')
fig_persona.show()

# Foco nos Campeões por Cidade
campeoes = df_geo[df_geo['Persona'] == 'Campeões']
top_campeoes_cidades = campeoes['cidade'].value_counts().head(10).reset_index()
top_campeoes_cidades.columns = ['Cidade', 'Qtd_Campeões']

fig_vip = px.bar(top_campeoes_cidades, x='Cidade', y='Qtd_Campeões', 
                 title='Top 10 Cidades com Mais Clientes "Campeões"',
                 color='Qtd_Campeões', color_continuous_scale='Gold')
fig_vip.show()

## 6. Análise de Penetração Regional

Calcular o faturamento total por estado.

In [ ]:
fat_estado = df_geo.groupby('estado')['Monetario'].agg(['sum', 'mean']).reset_index()
fat_estado.columns = ['Estado', 'Faturamento_Total', 'Faturamento_Medio']
fat_estado = fat_estado.sort_values(by='Faturamento_Total', ascending=False)

fig3 = px.bar(fat_estado, x='Estado', y='Faturamento_Total',
              title='Faturamento Total por Estado',
              hover_data=['Faturamento_Medio'],
              color='Faturamento_Total', color_continuous_scale='Blues')
fig3.show()

## 7. Conclusões Geográficas

- **Concentração:** Identificamos as cidades e estados com maior volume de clientes.
- **Personas:** A análise Sunburst e o gráfico de VIPs mostram se certas regiões têm maior propensão a clientes de alto valor.
- **Ação:** Estes dados podem ser usados para direcionar campanhas de marketing localizadas ou otimização logística.